This code creates three tables.  

- ratings - which contains the history from 1990 of S&P's corporate ratings (and maybe some other types)
- financials - some financial records, not complete, from compustat from 1990
- info - industry classifications and region/state information

In [77]:
# This is the only place that the connection is made.  It sometimes times out.

%run '/Users/ghost/Desktop/code snippets/standard imports.py'

overwrite = True

if overwrite:
    import wrds
    conn = wrds.Connection(wrds_username='ghostmann')
else:
    print("Data will be pulled from previous downloads, saved locally.\n")


Importing pandas as pd, numpy as np, datetime as dt, matplotlib.pyplot as plt
Setting Matplotlib style to Five Thirty Eight 

Loading library list...
Done


In [78]:
# downloading a identifier conversion table so we can splice in gvkeys

if overwrite:
    query = """ SELECT * FROM ciq.wrds_gvkey  """
    gvkey = conn.raw_sql(query)
    gvkey.startdate = pd.to_datetime(gvkey.startdate)
    gvkey.enddate = pd.to_datetime(gvkey.enddate)
    gvkey = gvkey.rename({'companyid':'company_id', 'companyname':'name', 'startdate':'start_date', 'enddate':'end_date', 
                          'primaryflag':'primary_flag'}, axis = 1)
    gvkey.loc[gvkey.start_date.isna(), 'start_date'] = pd.Timestamp(1901, 1, 1)
    gvkey.loc[gvkey.end_date.isna(), 'end_date'] = pd.Timestamp(2199, 1, 1)

    gvkey.to_parquet('/Users/ghost/Desktop/Credit/data/gvkeys.parquet')
else:
    print('\nPulling local data.')
    gvkey = pd.read_parquet('/Users/ghost/Desktop/Credit/data/gvkeys.parquet')

print(f'\nThe number of rows is {len(gvkey):,}.\n')
gvkey.head()


The number of rows is 140,638.



,company_id,gvkey,name,start_date,end_date,primary_flag
0,18507.0,235716,2M Invest A/S,1901-01-01,2199-01-01,1
1,18511.0,210835,3i Group plc,1901-01-01,2199-01-01,1
2,18527.0,210418,ABB Ltd,1901-01-01,2199-01-01,1
3,18671.0,029751,Albemarle Corporation,1901-01-01,2199-01-01,1
4,18711.0,028349,The Allstate Corporation,1901-01-01,2199-01-01,1


In [79]:
# this is where we pull down the financials for use in modeling
# we should add anything additional that we can think of

if overwrite:

    sql_financials = """
    SELECT 
        gvkey, 
        datadate,          -- Statement Date (Fiscal Year-End)
        fyear,             -- Fiscal Year
        fyr,               -- Fiscal Year-End Month
        at,                -- Total Assets
        lt,                -- Total Liabilities
        ceq,               -- Common Equity
        act,               -- Current Assets
        lct,               -- Current Liabilities
        invt,              -- Inventories
        rect,              -- Accounts Receivable
        ap,                -- Accounts Payable
        dlc,               -- Debt in Current Liabilities (Short-Term Debt)
        dltt,              -- Debt in Long-Term Liabilities
        dltis,             -- Debt Issued
        dvt,               -- Debt Valuation
        che,               -- Cash and Short-Term Investments
        xint,              -- Interest Expense
        xrd,               -- Research and Development Expense
        xsga,              -- Selling, General, and Administrative Expenses
        oibdp,             -- Operating Income Before Depreciation
        ebit,              -- Earnings Before Interest and Taxes
        ebitda,            -- Earnings Before Interest, Taxes, Depreciation, and Amortization
        sale,              -- Sales/Revenue
        cogs,              -- Cost of Goods Sold
        ni,                -- Net Income
        oancf,             -- Operating Cash Flow
        fincf,             -- Financing Cash Flow
        csho,              -- Common Shares Outstanding
        prcc_f,            -- Price Close, fiscal year
        prcc_c,            -- Price Close, calendar year
        'Annual' AS freq,  -- Data Frequency Indicator
        pstk,              -- Preferred Stock
        dvp,              -- Preferred Stock Cash Dividends
        dp,                -- Depreciation
        capx,              -- Capital Expenditures
        intan,             -- Intangible Assets
        gdwl,              -- Goodwill
        txt,               -- Tax Expense
        dv,                -- Dividends (if available)
        urect             -- net account receivables
    FROM 
        comp.funda
    WHERE 
        indfmt = 'INDL'     -- Industrial Format
        AND datafmt = 'STD' -- Standardized Format
        AND popsrc = 'D'     -- Domestic Companies
        AND consol = 'C'     -- Consolidated Data
    
        AND fyear >= 1990    -- Adjust the starting fiscal year as needed
    """
    
    financials = conn.raw_sql(sql_financials)
    financials.datadate = pd.to_datetime(financials.datadate)
    financials = financials.rename({'datadate':'data_date'}, axis = 1)
    financials.to_parquet('/Users/ghost/Desktop/Credit/data/compustat financials.parquet')
else:
    print('\nPulling local data.')
    financials = pd.read_parquet('/Users/ghost/Desktop/Credit/data/compustat financials.parquet')

print(f'\nThe number of rows is {len(financials):,}.')
print(f'The final date is {financials.data_date.max()}.\n')
financials.head()


The number of rows is 410,129.
The final date is 2025-12-31 00:00:00.



,gvkey,data_date,fyear,fyr,at,lt,ceq,act,lct,invt,...,freq,pstk,dvp,dp,capx,intan,gdwl,txt,dv,urect
0,001004,1991-05-31,1990,5,379.958,186.180,193.778,268.399,79.227,156.133,...,Annual,0.0,0.0,8.256,8.884,7.115,7.115,6.550,7.651,NaN
1,001009,1990-10-31,1990,10,32.335,26.073,6.262,10.047,8.382,2.355,...,Annual,0.0,0.0,1.971,5.493,0.000,0.000,0.769,0.808,NaN
2,001010,1990-12-31,1990,12,1728.888,1516.287,210.399,523.502,421.777,36.860,...,Annual,0.0,0.0,56.747,140.385,NaN,0.000,-28.220,0.000,NaN
3,001011,1990-12-31,1990,12,7.784,7.117,0.667,1.247,1.246,0.663,...,Annual,0.0,0.0,0.700,1.273,1.009,NaN,0.000,0.000,NaN
4,001013,1990-10-31,1990,10,181.665,47.652,134.013,102.525,37.335,33.845,...,Annual,0.0,0.0,11.980,13.734,NaN,NaN,15.269,0.000,NaN


In [80]:
# this is where we get information on industry and region

if overwrite:

    sql_info = """
    SELECT 
        gvkey,
        conm,       -- Company Name
        fic,        -- Current ISO Country Code - Incorporation
        fyrc,       -- month of fiscal year end
        gsector,    -- GIC Sector  (Global Identification Code)
        ggroup,     -- GIC Group
        gind,       -- GIC Industries
        idbflag,    -- International, Domestic, Both Indicator
        incorp,     -- Current State/Province of Incorporation Code
        loc,        -- Current ISO Country Code - Headquarters
        naics,      -- North American Industry Classification Code
        sic,        -- Standard Identification Code
        state       -- State/Province
    FROM 
        comp.company
    """
    # Note that GICs is the current industry leader while SIC and NAICs are considered outdated
    
    info = conn.raw_sql(sql_info)
    info.to_parquet('/Users/ghost/Desktop/Credit/data/company info.parquet')
else:
    print('\nPulling local data.')
    info = pd.read_parquet('/Users/ghost/Desktop/Credit/data/company info.parquet')
    
print(f'\nThe number of rows is {len(info):,}.\n')
info.head()


The number of rows is 56,754.



,gvkey,conm,fic,fyrc,gsector,ggroup,gind,idbflag,incorp,loc,naics,sic,state
0,001000,A & E PLASTIK PAK INC,USA,12,None,None,None,D,None,USA,None,3089,None
1,001001,A & M FOOD SERVICES INC,USA,12,25,2530,253010,D,None,USA,722,5812,OK
2,001002,AAI CORP,USA,12,None,None,None,D,None,USA,None,3825,MD
3,001003,A.A. IMPORTING CO INC,USA,1,25,2550,255040,D,DE,USA,442110,5712,MO
4,001004,AAR CORP,USA,5,20,2010,201010,D,DE,USA,423860,5080,IL


The following is intended to clean up the ratings data for use in actual analysis.

In [81]:
# This block pulls down all the S&P ratings. 

if overwrite:
    query = """
    SELECT *
    FROM ciq_ratings.wrds_erating
    WHERE ratingdate >= '1970-01-01'
    """
    ratings = conn.raw_sql(query)
    ratings.ratingdate = pd.to_datetime(ratings.ratingdate)
    ratings.creditwatchdate = pd.to_datetime(ratings.creditwatchdate)
    ratings.outlookdate = pd.to_datetime(ratings.outlookdate)
    
    ratings = pd.merge(gvkey[['gvkey', 'company_id', 'start_date', 'end_date']], ratings, on = 'company_id')

    ratings = ratings[ratings.ratingdate >= ratings.start_date]
    ratings = ratings[ratings.ratingdate <= ratings.end_date]

    ratings = ratings[ratings.orgdebttypecode.isin(['ICR', 'FSR'])]
    ratings = ratings[ratings.ratingtypecode == 'STDLONG']
    ratings = ratings[~ratings.ratingqualifier.isin(['pi', 'q'])]
    ratings = ratings[~ratings.ratingsymbol.str.contains('/')]

    ratings['outlook'] = ratings['outlook'].replace({'NM': 'NR'})
    ratings['creditwatch'] = ratings['creditwatch'].replace({'NM': 'NR'})

    ratings.to_parquet('/Users/ghost/Desktop/Credit/data/S&P ratings raw.parquet')
else:
    print('\nPulling local data.')
    ratings = pd.read_parquet('/Users/ghost/Desktop/Credit/data/S&P ratings raw.parquet')

print(f'\nThe number of rows is {len(ratings):,}.')
print(f'The final date is {ratings.ratingdate.max()}.\n')

ratings.head()


The number of rows is 92,591.
The final date is 2025-12-23 00:00:00.



,gvkey,company_id,start_date,end_date,entity_id,entity_pname,ratingdate,ratingdetailid,orgdebttypecode,ratingtypecode,...,creditwatchtime,creditwatchdate,outlooktime,outlookdate,unsol,ratingtypename,longtermflag,shorttermflag,globalornationalscaleind,ratingtypeid
8,210835,18511.0,1901-01-01,2199-01-01,111802,3i Group PLC,1995-12-07,70816.0,ICR,STDLONG,...,None,NaT,17:30:35,1995-12-07,N,Local Currency LT,1.0,0.0,G,130
10,210835,18511.0,1901-01-01,2199-01-01,111802,3i Group PLC,1995-12-07,7770208.0,ICR,STDLONG,...,05:09:54,2001-10-31,05:09:54,2001-10-31,N,Local Currency LT,1.0,0.0,G,130
12,210835,18511.0,1901-01-01,2199-01-01,111802,3i Group PLC,2001-12-20,7889042.0,ICR,STDLONG,...,10:35:06,2001-12-20,10:35:06,2001-12-20,N,Local Currency LT,1.0,0.0,G,130
16,210835,18511.0,1901-01-01,2199-01-01,111802,3i Group PLC,2001-12-20,8804964.0,ICR,STDLONG,...,None,NaT,07:58:25,2003-01-16,N,Local Currency LT,1.0,0.0,G,130
18,210835,18511.0,1901-01-01,2199-01-01,111802,3i Group PLC,2001-12-20,9879974.0,ICR,STDLONG,...,None,NaT,05:32:34,2004-02-11,N,Local Currency LT,1.0,0.0,G,130


In [82]:
# some reorganization
ratings = ratings.rename({'ratingdate':'rtg_date', 'ratingsymbol':'rtg_symbol',
                          'orgdebttypecode':'debt_type', 'entity_pname':'name'}, axis = 1)

ratings['OL_date'] = ratings.creditwatchdate
ratings.loc[ratings.cwolind == 'OL', 'OL_date'] = ratings.loc[ratings.cwolind == 'OL', 'outlookdate']
ratings['guidance'] = ratings.creditwatch
ratings.loc[ratings.cwolind == 'OL', 'guidance'] = ratings.loc[ratings.cwolind == 'OL', 'outlook']

ratings = ratings.drop(['ratingqualifier', 'longtermflag', 'shorttermflag', 'cwolactionword',
            'globalornationalscaleind', 'currentratingsymbol', 'ratingdetailid', 
            'ratingtime', 'creditwatchtime', 'outlooktime', 'ratingtypename',
            'regulatoryindicator', 'ratingtypeid', 'start_date', 'end_date', 'unsol', 
            'ratingtypecode', 'priorcreditwatch', 'prioroutlook', 'priorratingsymbol',
            'company_id', 'entity_id', 'outlookdate', 'creditwatchdate', 'outlook', 'creditwatch',
            'ratingactionword', 'cwolind'],
           axis = 1)

ratings = ratings.sort_values(['gvkey', 'rtg_date', 'OL_date'])

# creating a list of names, keeping only the last name chronologically
names = ratings[['gvkey', 'name', 'rtg_date']].drop_duplicates()
names.columns = ['gvkey', 'name', 'date']
b = names.groupby('gvkey', as_index = False).date.max()
names = pd.merge(names, b, on = ['gvkey', 'date'])
names = names.groupby('gvkey', as_index = False).last()
del b, names['date']
ratings.head()

,gvkey,name,rtg_date,debt_type,rtg_symbol,OL_date,guidance
34641,001004,AAR Corp.,1987-05-28,ICR,BBB,1988-11-04,Stable
34642,001004,AAR Corp.,1987-05-28,ICR,BBB,1993-03-10,Negative
34638,001004,AAR Corp.,1987-05-28,ICR,BBB,NaT,None
34644,001004,AAR Corp.,1995-01-23,ICR,BBB-,1995-01-23,Stable
34646,001004,AAR Corp.,1995-01-23,ICR,BBB-,1997-02-07,Positive


In [83]:
# These next two blocks condense the histories so we don't see the same rating/outlook
# over and over sequentially.  It doesn't matter but makes troubleshooting easier

only_ratings = ratings[['gvkey', 'rtg_date', 'rtg_symbol']] \
    .drop_duplicates() \
    .sort_values(['gvkey', 'rtg_date'])
is_new_rating = list(only_ratings.groupby('gvkey')['rtg_symbol'].apply(lambda s: s.ne(s.shift(1))))
only_ratings = only_ratings[is_new_rating]
only_ratings['end_date'] = only_ratings.groupby('gvkey')['rtg_date'].shift(-1)
only_ratings.loc[only_ratings['end_date'].isna(), 'end_date'] = pd.Timestamp(2199, 12, 31)  # open-ended tail

outlooks = ratings[['gvkey', 'OL_date', 'guidance']] \
    .drop_duplicates() \
    .sort_values(['gvkey', 'OL_date']) \
    .dropna()
is_new_outlook = list(outlooks.groupby('gvkey')['guidance'].apply(lambda s: s.ne(s.shift(1))))
outlooks = outlooks[is_new_outlook]
outlooks['end_date'] = outlooks.groupby('gvkey')['OL_date'].shift(-1)
outlooks.loc[outlooks['end_date'].isna(), 'end_date'] = pd.Timestamp(2199, 12, 31)  # open-ended tail

# The rest of the blocks in the cell make sure that a rating of NR corresponds 
# an outlook of NR.  It's not complete yet because this is a difficult problem.

NRs = only_ratings[only_ratings.rtg_symbol == 'NR'] \
    .drop(['rtg_symbol'], axis = 1) \
    .rename({'end_date':'NR_end_date', 'rtg_date':'NR_date'}, axis = 1)
a = pd.merge(outlooks, NRs, how = 'left', on = 'gvkey') \
    .dropna()

b = a[(a.guidance != 'NR') & (a.OL_date < a.NR_date) & (a.end_date > a.NR_date)]
outlooks = pd.merge(outlooks, b[['gvkey', 'OL_date', 'NR_date']], \
                    on = ['gvkey', 'OL_date'], how = 'left')
outlooks.loc[~outlooks.NR_date.isna(), 'end_date'] = outlooks.loc[~outlooks.NR_date.isna(), 'NR_date']
outlooks = outlooks.drop(['NR_date'], axis = 1)

b = a[(a.guidance != 'NR') & (a.OL_date < a.NR_end_date) & (a.end_date > a.NR_end_date)]
outlooks = pd.merge(outlooks, b[['gvkey', 'OL_date', 'NR_end_date']], \
                    on = ['gvkey', 'OL_date'], how = 'left')
outlooks.loc[~outlooks.NR_end_date.isna(), 'OL_date'] \
            = outlooks.loc[~outlooks.NR_end_date.isna(), 'NR_end_date']
outlooks = outlooks.drop(['NR_end_date'], axis = 1)

b = a[(a.guidance != 'NR') & (a.OL_date >= a.NR_date) & (a.end_date <= a.NR_date)]
outlooks = pd.merge(outlooks, b[['gvkey', 'OL_date', 'NR_end_date']], \
                    on = ['gvkey', 'OL_date'], how = 'left')
outlooks.loc[~outlooks.NR_end_date.isna(), 'guidance'] = 'NR'
outlooks = outlooks.drop(['NR_end_date'], axis = 1)

outlooks.head()

,gvkey,OL_date,guidance,end_date
0,001004,1988-11-04,Stable,1993-03-10
1,001004,1993-03-10,Negative,1995-01-23
2,001004,1995-01-23,Stable,1997-02-07
3,001004,1997-02-07,Positive,1997-07-03
4,001004,1997-07-03,Stable,2001-09-21


In [84]:
only_ratings.columns = ['gvkey', 'date', 'rtg_symbol', 'end_date']
outlooks.columns = ['gvkey', 'date', 'guidance', 'end_date']

data = pd.concat([only_ratings[['gvkey', 'date']], outlooks[['gvkey', 'date']]]) \
    .drop_duplicates() \
    .sort_values(['gvkey', 'date'])
only_ratings.columns = ['gvkey', 'rtg_date', 'rtg_symbol', 'end_date']

data = pd.merge(data, only_ratings, on = 'gvkey', how = 'left')
data = data[(data.rtg_date <= data.date) & (data.end_date > data.date)] \
    .drop(['rtg_date', 'end_date'], axis = 1)

outlooks.columns = ['gvkey', 'OL_date', 'guidance', 'end_date']
merged = pd.merge(data, outlooks, on = 'gvkey', how = 'left')
merged = merged[(merged.date >= merged.OL_date) & (merged.date < merged.end_date)]

data = pd.merge(data, merged[['gvkey', 'date', 'guidance']], on = ['gvkey', 'date'], how = 'left')
data = data.sort_values(['gvkey', 'date'])

data['end_date'] = data.groupby('gvkey')['date'].shift(-1)
data.loc[data['end_date'].isna(), 'end_date'] = pd.Timestamp(2199, 12, 31)  # open-ended tail

rating_map = {'AAA':1, 'AA+':2,  'AA':3, 'AA-':4,  'A+':5, 'A':6, 'A-':7, 'BBB+':8, 'BBB':9, 'BBB-':10,
              'BB+':11, 'BB':12, 'BB-':13, 'B+':14, 'B':15, 'B-':16, 'CCC+':17, 'CCC':18, 'CCC-':19,
              'CC':20, 'C':21, 'D':100, 'SD':100, 'R':100, 'NR':0}
data['rtg_number'] = data.rtg_symbol.map(rating_map)

data = data[['gvkey', 'date', 'end_date', 'rtg_symbol', 'rtg_number', 'guidance']]
data = data.sort_values(['gvkey', 'date']) \
           .reset_index(drop = True)

print(f'\nThe number of rows is {len(data):,}.')
print(f'The final date is {data.date.max()}.\n')
data.head()


The number of rows is 89,451.
The final date is 2025-12-23 00:00:00.



,gvkey,date,end_date,rtg_symbol,rtg_number,guidance
0,001004,1987-05-28,1988-11-04,BBB,9,NaN
1,001004,1988-11-04,1993-03-10,BBB,9,Stable
2,001004,1993-03-10,1995-01-23,BBB,9,Negative
3,001004,1995-01-23,1997-02-07,BBB-,10,Stable
4,001004,1997-02-07,1997-07-03,BBB-,10,Positive


In [85]:
data.to_parquet('/Users/ghost/Desktop/Credit/data/S&P ratings.parquet')